## FILE PATHS

In [62]:
SEC_JSON_PATH = 'intemediate_data/sec_reg_structured.json'
SEC_W_KW_PATH = 'intemediate_data/sec_reg_tree_enhanced.json'
QUESTION_JSON_PATH = 'questions/eval_questions.json'
TREERAG_RESULT_PATH = 'results/legaltreerag_ans.csv'

## LLM MODELS

In [63]:
# LLM_MODEL_RETRIEVAL = "openai/gpt-oss-120b"
# LLM_MODEL_ANS = "openai/gpt-oss-120b"


LLM_MODEL_RETRIEVAL = "openai/gpt-4o"
LLM_MODEL_ANS = "openai/gpt-4o"

In [64]:
OPENROUTER_MODEL_URL = "https://openrouter.ai/api/v1"
GITHUB_MODEL_URL = "https://models.github.ai/inference"
GROQ_MODEL_URL = "https://api.groq.com/openai/v1/models"

In [65]:
import os
from dotenv import load_dotenv

# โหลดค่าจากไฟล์ .env
load_dotenv()

# ดึงค่ามาใช้
GROQ_KEY = os.getenv("GROQ_KEY")
OPENROUTER_KEY = os.getenv("OPENROUTER_KEY")
GITHUB_KEY = os.getenv("GITHUB_KEY")

In [67]:
from openai import OpenAI
from groq import Groq
import time
import json
import re

In [68]:

# os.environ["GROQ_API_KEY"] = GROQ_KEY
# client = Groq()

In [69]:


# client = OpenAI(base_url=OPENROUTER__MODEL_URL, api_key=OPENROUTER_KEY,)
client = OpenAI(base_url=GITHUB_MODEL_URL, api_key=GITHUB_KEY)


## Phase 1 : Regulations Retrieval

## 1 Agentic Routing

In [70]:
import json
import re
from groq import Groq
import os
import time


# 1. Load Tree
with open(SEC_W_KW_PATH, 'r', encoding='utf-8') as f:
    reg_data = json.load(f)

# 2. Build Node Index
node_index = {}
def build_index(nodes):
    for node in nodes:
        if 'node_id' in node:
            node_index[node['node_id']] = node
        if 'subsection' in node and len(node['subsection']) > 0:
            build_index(node['subsection'])

build_index(reg_data['data'])

# 3. Create Table of COntent from Tree 
def generate_toc_string_with_desc(nodes, depth=0):
    toc = ""
    exclude_types = ['ข้อย่อย', 'อนุย่อย', 'ส่วนนำ']
    for node in nodes:
        sec_type = node.get('section_type', '')
        if sec_type in exclude_types:
            continue
            
        node_id = node.get('node_id')
        full_path = " ".join(node.get('full_context_path', '').split())
        
        desc = " ".join(node.get('description', '').split())
        short_desc = (desc[:80] + '...') if len(desc) > 80 else desc
        desc_text = f" ({short_desc})" if short_desc else ""
        
        keywords = node.get('keywords', '')
        keyword_text = f" [Keywords: {keywords}]" if keywords else ""
        
        toc += f"- [{node_id}] {full_path}{desc_text}{keyword_text}\n"
        
        if 'subsection' in node and len(node['subsection']) > 0:
            toc += generate_toc_string_with_desc(node['subsection'], depth + 1)
            
    return toc

# ------------------------------------------------------------------
# 4. Helper for LLM 
# ------------------------------------------------------------------
def call_router_llm(query, partial_toc, round_num):


    prompt = f"""
    คุณคือผู้เชี่ยวชาญด้านกฎหมาย หน้าที่ของคุณคือการหาหัวข้อกฎหมายที่ซ่อนอยู่ใน "สารบัญโครงสร้าง" เพื่อนำไปตอบคำถาม
    
    คำถามของผู้ใช้: "{query}"
    
    สารบัญส่วนที่ {round_num} (รูปแบบ: - [ID] เส้นทางหัวข้อ (คำอธิบาย) [Keywords: ...]):
    {partial_toc}
    
    ขั้นตอนการทำงาน:
    1. วิเคราะห์คำถามและสกัด "Keywords สำคัญ" ออกมาให้ครบถ้วน เน้นสั้นๆ จำนวนเยอะ โดยเฉพาะชื่อต่างๆ
    2. จากข้อมูลในสารบัญ "ให้รวบรวม ID" ที่เกี่ยวข้องกับ keywords ที่ได้จากข้อ 1 มาให้ครบ
    3. เทียบข้อมูลของ ID ทั้งหมด จากข้อ 2 กับ คำถามของผู้ใช้ เพื่อคัดเลือก ID ที่เกี่ยวข้องออกมาเป็น Array
    3. หาก "ไม่พบ" หัวข้อที่เกี่ยวข้องในสารบัญ ให้ตอบ Array ว่าง []
    
    ข้อบังคับการตอบ (สำคัญมาก):
    - ให้อธิบายกระบวนการคิดสั้นๆ ว่าพบ Keyword ที่ ID ใดบ้าง
    - ห้ามตัด ID ใดทิ้งเด็ดขาด หากคุณอ้างถึง ID ใดในคำอธิบาย ID นั้นต้องปรากฏใน Array ด้วย
    - "การคัดลอก ID": ห้ามพิมพ์รหัส ID ใหม่เอง ให้คัดลอกรหัส ID จากสารบัญมาวางเท่านั้น
    - "บรรทัดสุดท้าย" ต้องเป็น Array ของรหัส ID เท่านั้น เช่น ["root_2_1", "root_3_2_2_5"] หรือ [] 
    """

    
    try:
        response = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model=LLM_MODEL_RETRIEVAL,
            temperature=0.0,
        )
        reply = response.choices[0].message.content.strip()
        print(f"🧠 [Agent Thinking - รอบที่ {round_num}]:\n{reply}\n")
        
        matches = re.findall(r'\[.*?\]', reply, re.DOTALL)
        if matches:
            clean_match = matches[-1].replace("'", '"') 
            clean_match = clean_match.replace(".", "_")
            return json.loads(clean_match)
        return []
    except Exception as e:
        print(f"Error Routing รอบ {round_num}: {e}")
        return []


# ------------------------------------------------------------------
# 5. Main: divide Node to 2 pieces
# ------------------------------------------------------------------
def get_relevant_nodes_two_pass(query, nodes):
    mid_point = int(len(nodes) // 1.5)
    batch_1 = nodes[:mid_point]
    batch_2 = nodes[mid_point:]
    
    toc_1 = generate_toc_string_with_desc(batch_1)
    toc_2 = generate_toc_string_with_desc(batch_2)
    
    all_selected_ids = []
    
    print("🚀 เริ่มสแกนรอบที่ 1/2 (ครึ่งแรก)...")
    ids_1 = call_router_llm(query, toc_1, round_num=1)
    all_selected_ids.extend(ids_1)
    # time.sleep(2)
    
    print("🚀 เริ่มสแกนรอบที่ 2/2 (ครึ่งหลัง)...")
    ids_2 = call_router_llm(query, toc_2, round_num=2)
    all_selected_ids.extend(ids_2)
    
    return list(set(all_selected_ids))

# ------------------------------------------------------------------
# 🎯 6. (Small-to-Big)
# ------------------------------------------------------------------

def expand_to_parent_context(selected_ids, node_index):
    expanded_ids = set()
    
    # ระดับที่อนุญาตให้ดึงข้อมูลทั้งก้อนได้ (ไม่ล้นเกินไป)
    allowed_parent_types = ['หมวด', 'ส่วน', 'คำศัพท์'] 
    
    # 📌 ระดับที่ใหญ่เกินไป ถ้า LLM เผลอเลือกมา ให้ "ตัดทิ้ง" ออกจาก Context ทันที
    exclude_types = ['ภาค', 'บทนำ', 'ส่วนนำ'] 
    
    for node_id in selected_ids:
        if node_id not in node_index:
            continue
            
        # 1. เช็คตั้งแต่ต้นทางเลยว่า LLM ดันไปจิ้มเลือก "ภาค" มาตรงๆ หรือเปล่า
        initial_type = node_index[node_id].get('section_type', '')
        if initial_type in exclude_types:
            print(f"✂️ ข้าม Node ต้นทางที่ใหญ่เกินไป: {node_id} (ระดับ: {initial_type})")
            continue # ข้าม ID นี้ไปเลย ไม่เก็บเข้า expanded_ids
            
        current_id = node_id
        original_node_id = node_id 
        
        # 2. ลูปเพื่อไต่ระดับขึ้นไปตามโครงสร้าง ID (สำหรับ Node ที่รอดด่านแรกมา)
        while current_id in node_index:
            node = node_index[current_id]
            sec_type = node.get('section_type', '')
            
            # ถ้าเป็นระดับเล็ก ให้ตัดหางเพื่อไต่ขึ้นไปหาแม่
            if sec_type in ['ข้อ', 'ข้อย่อย', 'อนุย่อย']:
                parts = current_id.split('_')
                if len(parts) > 1:
                    current_id = "_".join(parts[:-1]) # เลื่อนขึ้น 1 ระดับ
                else:
                    expanded_ids.add(original_node_id) # ต้นทางสุดแล้ว ให้เก็บข้อตัวเอง
                    break
                    
            # ถ้าไต่ขึ้นมาเจอระดับที่พอดี (หมวด, ส่วน) ให้ยึด ID ระดับนี้เลย
            elif sec_type in allowed_parent_types:
                expanded_ids.add(current_id)
                break
                
            # ถ้าไต่จากข้อขึ้นมาจนชน "ภาค" ให้ถอยกลับไปใช้ ID เดิม (ข้อเล็กๆ)
            else:
                expanded_ids.add(original_node_id) 
                break
                
    return list(expanded_ids)
    

In [71]:
full_toc = generate_toc_string_with_desc(reg_data['data'])

In [72]:
full_toc

'- [root_0]  (ประกาศคณะกรรมการกำกับตลาดทุน ที่ ทจ. 16/2565 เรื่อง การขออนุญาตและการอนุญาตให้เส...)\n- [root_0_1]  (ประกาศนี้ให้ใช้บังคับตั้งแต่วันที่ 1 ตุลาคม พ.ศ. 2565 เป็นต้นไป) [Keywords: วันที่ 1 ตุลาคม พ.ศ. 2565]\n- [root_0_2]  (เว้นแต่ประกาศนี้และภาคผนวกท้ายประกาศนี้ได้กำหนดไว้เป็นประการอื่นให้นำบทนิยามที่ก...) [Keywords: ตราสารหนี้, กองทรัสต์, หุ้นกู้แปลงสภาพ, หุ้นรองรับ, หุ้นกู้ด้อยสิทธิ, ข้อกำหนดสิทธิ]\n- [root_0_2_1]  (ตราสารหนี้ที่ออกใหม่ดังต่อไปนี้) [Keywords: ตราสารหนี้]\n- [root_0_2_2]  [Keywords: กองทรัสต์]\n- [root_0_2_3]  (หุ้นกู้ที่อาจแปลงสภาพแห่งสิทธิเป็นหุ้นที่ออกใหม่ของผู้ออกหุ้นกู้แปลงสภาพนั้น) [Keywords: หุ้นกู้แปลงสภาพ]\n- [root_0_2_4]  (หุ้นออกใหม่ที่จัดไว้เพื่อรองรับการแปลงสภาพ) [Keywords: หุ้นรองรับ]\n- [root_0_2_5]  (หุ้นกู้ไม่มีประกันที่มีการกำหนดสิทธิของผู้ถือหุ้นกู้ไว้ด้อยกว่าสิทธิของเจ้าหนี้ส...) [Keywords: หุ้นกู้ด้อยสิทธิ]\n- [root_0_2_6]  (ข้อกำหนดว่าด้วยสิทธิและหน้าที่ของผู้ออกตราสาร และผู้ถือตราสาร) [Keywords: ข้อกำหนดสิทธิ]\n- [root_0_2_7]  (สมาคมต

## Phase 2 The Generator: Summarizing the answer from the query regulation and question

In [73]:

import json

def generate_final_answer(query, selected_ids, index_data):
    """
    Phase 2: นำ Context ที่ดึงมาได้ มาให้ LLM อ่านและสรุปคำตอบ (รูปแบบ JSON)
    """
    if not selected_ids:
        return {
            "generated_conclusion": "ขออภัยครับ ไม่พบหัวข้อกฎหมายที่เกี่ยวข้องกับคำถามของคุณเลยครับ",
            "reference_clause": []
        }
        
    # 📌 ส่วนที่เพิ่มเข้ามา: กรอง Node ลูกทิ้ง ถ้ามี Node แม่อยู่ในลิสต์แล้ว
    filtered_ids = []
    for current_id in selected_ids:
        is_child = False
        for other_id in selected_ids:
            # ถ้า ID ปัจจุบันยาวกว่า และขึ้นต้นด้วย ID อื่น + "_" แสดงว่าเป็นลูก
            if current_id != other_id and current_id.startswith(other_id + "_"):
                is_child = True
                break
        
        # ถ้าไม่ใช่ลูก (ไม่มีแม่อยู่ในลิสต์) ให้เก็บไว้ใช้งาน
        if not is_child:
            filtered_ids.append(current_id)

    # พิมพ์ให้ดูว่ามีการกรองทิ้งไปกี่ Node
    if len(filtered_ids) < len(selected_ids):
        print(f"✂️ ตัด Node ซ้ำซ้อนทิ้ง: จาก {selected_ids} เหลือแค่ {filtered_ids}")
        
    # 1. รวบรวมเนื้อหา (Context) จาก Node ที่ผ่านการกรองแล้ว (ใช้ filtered_ids)
    context_text = ""
    for idx in filtered_ids:
        if idx in index_data:
            node = index_data[idx]
            
            # 📌 ดึงข้อมูล parent (ถ้ามี) แล้วเอามาต่อกับ section ปัจจุบัน
            parents = node.get('parent', [])
            current_section = node.get('section', '')
            
            if parents:
                # ถ้ามีแม่ ให้นำมาเชื่อมกันด้วย " > " (เช่น ภาค 1 > หมวด 1 > ข้อ 15)
                lineage = " > ".join(parents) + f" > {current_section}"
            else:
                # ถ้าไม่มีแม่ (อยู่ระดับบนสุด)
                lineage = current_section
                
            context_text += f"\n--- อ้างอิงจากโครงสร้าง: {lineage} ---\n"
            context_text += node.get('raw', node.get('description', '')) + "\n"
            
    # 2. สร้าง Prompt สำหรับ Phase 3
    prompt = f"""
    คุณคือนักกฎหมายผู้เชี่ยวชาญ หน้าที่ของคุณคือการตอบคำถามโดยอิงจาก "ข้อมูลอ้างอิง" ที่ให้มาเท่านั้น
    
    คำถาม: "{query}"
    
    ข้อมูลอ้างอิง:
    {context_text}
    
    ขั้นตอนการคิดและข้อควรระวัง (สำคัญมาก):
    1. ค้นหาและคัดกรองข้อกฎหมาย: ให้อ่านข้อมูลอ้างอิงทั้งหมด แล้วพิจารณาว่าข้อใดบ้างที่มีคำตอบของคำถามนี้ซ่อนอยู่ จากนั้นให้สกัด "เลขข้อกฎหมาย" **เฉพาะข้อที่คุณจะนำเนื้อหามาใช้ตอบจริงๆ เท่านั้น** (หากมีข้อมูลอ้างอิงใดที่ให้มาแต่ไม่เกี่ยวกับคำตอบ ห้ามนำเลขข้อนั้นมาระบุเด็ดขาด) โดยพยายามระบุลึกถึงระดับ วรรค หรือ อนุมาตรา เท่าที่ปรากฏ (เช่น ข้อ 15 (2))
    2. วิเคราะห์รูปแบบคำถามเพื่อกำหนดวิธีตอบ:
       - หากเป็นคำถามปลายปิด (เช่น ได้หรือไม่ / ต้องทำหรือไม่): ให้ "ฟันธง" ในประโยคแรก (เช่น ได้ / ไม่ได้ / ต้องทำ) **และประโยคถัดมาต้องอธิบายเหตุผลและเงื่อนไขประกอบเสมอ** (ห้ามตอบสั้นๆ แค่คำเดียว)
       - หากเป็นคำถามปลายเปิด (เช่น อย่างไร / มีเงื่อนไขอะไรบ้าง / หมายความว่าอย่างไร): ให้อธิบายเนื้อหา เงื่อนไข หรือนิยามออกมาได้เลยโดย **ไม่ต้องฟันธง**
    3. ห้ามบิดเบือนตรรกะทางกฎหมายเด็ดขาด!: หากกฎหมายเขียนว่า "ห้ามทำ เว้นแต่..." ให้คุณตอบว่า "ทำไม่ได้ เว้นแต่..." ห้ามพลิกแพลงไปตอบว่า "ทำได้ ถ้า..." 
    
    รูปแบบการตอบ (ต้องตอบเป็น JSON Object เท่านั้น และให้ลำดับ Key ตามตัวอย่างนี้เพื่อบังคับกระบวนการคิด):
    {{
        "reference_clause": ["ข้อ 13 (1)", "ข้อ 54 (2)"], // 📌 ระบุเฉพาะข้อกฎหมายที่ใช้เป็นฐานของคำตอบจริงๆ เท่านั้น
        "generated_conclusion": "คำตอบที่ตรงประเด็น พร้อมอธิบายเหตุผลหรือเงื่อนไขตามข้อกฎหมายที่ดึงมาด้านบน..." 
    }}
    """
    
    print("🧠 กำลังให้ Model อ่านกฎหมายและสรุปคำตอบ (JSON Format)...\n")
    


    try:
        response = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model=LLM_MODEL_ANS,
            temperature=0.1, 
            response_format={"type": "json_object"} 
        )
        
        # 📌 1. ดึงค่า content ออกมาและตรวจสอบว่าเป็น None หรือไม่
        raw_content = response.choices[0].message.content
        
        if raw_content is None:
            finish_reason = response.choices[0].finish_reason
            print(f"⚠️ คำเตือน: API คืนค่าว่าง (อาจติด Safety Filter หรือ Error) Finish Reason: {finish_reason}")
            return {
                "generated_conclusion": "ขออภัยครับ AI ไม่สามารถสร้างคำตอบได้เนื่องจากข้อจำกัดของโมเดลหรือการกรองเนื้อหา",
                "reference_clause": []
            }
            
        reply = raw_content.strip()
        
        # 📌 2. ดักจับ Error กรณีโมเดลตอบกลับมามี Markdown (```json ... ```) ปนมาด้วย
        import re
        matches = re.findall(r'\{.*\}', reply, re.DOTALL)
        
        if matches:
             result_json = json.loads(matches[0])
        else:
             result_json = json.loads(reply)
             
        # 🟢 เพิ่มบรรทัดนี้: ยัด Context ที่ใช้ กลับเข้าไปใน output ด้วย
        result_json["retrieved_context"] = context_text.strip()

        return result_json
        
    except Exception as e:
        print(f"Error Generating Answer: {e}")
        return {
            "generated_conclusion": f"เกิดข้อผิดพลาดในการสร้างคำตอบ: {e}",
            "reference_clause": [],
            "retrieved_context": context_text.strip()
        }

# ==========================================
# 🚀 ทดสอบรัน Phase 3 
# ==========================================

# # สมมติว่านี่คือสิ่งที่เราได้มาจาก Phase 2
# test_query = "หากบริษัทต้องการเสนอขายพันธบัตรให้แก่กลุ่มเจ้าหนี้เดิมเพื่อ 'ปรับโครงสร้างหนี้' บริษัทนี้จะถูกบังคับให้ขายพันธบัตรให้เสร็จภายใน 6 เดือน เหมือนกับการเสนอขายทั่วไปหรือไม่?"
# # สมมติว่า Router ดึงข้อ 30 และ ข้อ 15 มาได้
# selected_ids = ['root_2_15_15', 'root_4_0_30'] # เปลี่ยนเป็น ID จริงในระบบของคุณ

# print("="*50)
# print("🎯 ผลลัพธ์สุดท้ายจาก Agentic Tree-RAG (Phase 3)")
# print("="*50)

# final_answer_data = generate_final_answer(test_query, selected_ids, node_index)

# # พิมพ์ผลลัพธ์แบบสวยงาม
# print(json.dumps(final_answer_data, indent=4, ensure_ascii=False))

## Phase 3: Running the Queries

In [74]:
import json
import time
import pandas as pd
from IPython.display import display, HTML

# ---------------------------------------------------------
# 1. Load Questions
# ---------------------------------------------------------
try:
    with open(QUESTION_JSON_PATH, 'r', encoding='utf-8') as f:
        eval_data = json.load(f)
    print(f"✅ โหลดข้อสอบสำเร็จ จำนวน {len(eval_data)} ข้อ")
except FileNotFoundError:
    print("❌ ไม่พบไฟล์ eval_questions2.json โปรดตรวจสอบชื่อไฟล์อีกครั้ง")
    eval_data = []

# ---------------------------------------------------------
# 2. Loop Agentic Tree-RAG
# ---------------------------------------------------------

# eval_data = eval_data[1:2]
# eval_data = eval_data[4:5]
# eval_data = eval_data[9:]

results = []
total_questions = len(eval_data)

print("\n🚀 Start Tree-RAG Evaluation Pipeline ...\n")

for i, item in enumerate(eval_data, 1):
    query = item['query']
    difficulty = item['difficulty']
    expected_ans = item.get('expected_answer', '')
    expected_clauses = item.get('reference_clause', [])
    
    print(f"\n📝 answering {i}/{total_questions} (Level: {difficulty})...")
    
    try:
        start_time = time.time()
        
        # 1. Routing
        selected_ids = get_relevant_nodes_two_pass(query, reg_data['data'])
        print(f"LLM Chooses Node (Raw): {selected_ids}")

        expanded_ids = expand_to_parent_context(selected_ids, node_index)
        print(f">>> Parent Node: {expanded_ids}\n")
        
        # 2. Generation 
        generated_data = generate_final_answer(query, expanded_ids, node_index)

        # จัดการข้อมูล clauses ให้อยู่ในรูปแบบ String ที่อ่านง่าย
        str_expected_clauses = ", ".join(expected_clauses) if isinstance(expected_clauses, list) else expected_clauses
        
        # ดึง reference_clause ที่ได้จาก Phase 3 
        gen_clauses_raw = generated_data.get('reference_clause', [])
        str_generated_clauses = ", ".join(gen_clauses_raw) if isinstance(gen_clauses_raw, list) else str(gen_clauses_raw)
        
        latency = time.time() - start_time
        
        # 3. บันทึกผลลัพธ์ (เพิ่ม Retrieved Context เพื่อนำไปหา Faithfulness ทีหลัง)
        results.append({
            "Q_ID": item.get('id', f"Q{i}"),
            "Difficulty": difficulty,
            "Latency (s)": round(latency, 2),
            "Expected Clauses": str_expected_clauses,           
            "Generated Clauses": str_generated_clauses,         
            "Expected Answer": expected_ans,                    
            "Generated Answer": generated_data.get('generated_conclusion', ''),
            "Retrieved Context": generated_data.get('retrieved_context', '') # 🟢 <--- เพิ่มบรรทัดนี้เข้ามาครับ!
        })
        
        if i < total_questions:
            print("⏳ rest for API...")
            time.sleep(5) 
        print("===============================================")
            
    except Exception as e:
        print(f"❌ Error in {i}: {e}")
        if "429" in str(e) or "413" in str(e):
            print("⏳ wait for API Error ...")
            time.sleep(60)

# ---------------------------------------------------------
# 3. Table
# ---------------------------------------------------------
df_results = pd.DataFrame(results)

# พิมพ์ให้ดูเพื่อความชัวร์ว่าคอลัมน์มาครบ
print(f"\n✅ ดำเนินการเสร็จสิ้น! สรุปผลตารางขนาด: {df_results.shape}")
# display(df_results.head())

✅ โหลดข้อสอบสำเร็จ จำนวน 10 ข้อ

🚀 เริ่มต้นการทดสอบ Tree-RAG Evaluation Pipeline (เตรียมพร้อมทำ Faithfulness)...


📝 กำลังทดสอบข้อที่ 1/1 (ระดับ: ยาก)...
🚀 เริ่มสแกนรอบที่ 1/2 (ครึ่งแรก)...
🧠 [Agent Thinking - รอบที่ 1]:
### การวิเคราะห์คำถามและสกัด Keywords
คำถามของผู้ใช้เกี่ยวข้องกับการพิจารณาคุณสมบัติของบริษัทมหาชนที่เคยถูกศาลพิพากษาคดีฉ้อโกงประชาชนเมื่อ 5 ปีก่อน ว่าจะขาดคุณสมบัติในการขออนุญาตเสนอขายตราสารหนี้ต่อผู้ลงทุนรายใหญ่หรือไม่

**Keywords สำคัญที่สกัดได้:**
- บริษัทมหาชน
- จดทะเบียน
- ตลาดหลักทรัพย์
- ศาลพิพากษา
- คดีฉ้อโกงประชาชน
- คุณสมบัติ
- การขออนุญาต
- การเสนอขายตราสารหนี้
- ผู้ลงทุนรายใหญ่

---

### การรวบรวม ID ที่เกี่ยวข้อง
จากการตรวจสอบสารบัญ พบหัวข้อที่เกี่ยวข้องกับ Keywords ดังนี้:

1. **[root_1_3_1_1]**: มีคำว่า "การไม่อนุญาต", "การเสนอขายตราสารหนี้", "ผลกระทบต่อความน่าเชื่อถือ", "ความเสียหายต่อผู้ลงทุน" ซึ่งอาจเกี่ยวข้องกับการพิจารณาคุณสมบัติของบริษัท
2. **[root_3_1_1_2]**: มีคำว่า "ผู้ออกตราสารหนี้", "ประวัติในเรื่องใดเรื่องหนึ่ง", "การเสนอขายตราสารหนี้ไม่ว่าป

In [75]:
df_results.to_csv(TREERAG_RESULT_PATH, index=False, encoding='utf-8-sig')